In [ ]:
from pathlib import Path
import polars as pl

# Run from the repository root (run_all.py sets this automatically).
for split in ("train", "validation", "test"):
    Path(split).mkdir(parents=True, exist_ok=True)
print("import success")

### Read MIND Training Dataset, Split into Train and Validation

In [ ]:
behaviors = pl.read_csv(
    "train/behaviors.tsv", 
    separator="\t", 
    has_header=False, 
    new_columns=["session_id", "user_id", "timestamp", "history", "impressions"]
)
print(behaviors.height)
behaviors.head()

In [ ]:
behaviors = behaviors.with_columns(
    pl.col("timestamp").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p")
).sort("timestamp")
behaviors.head()

In [ ]:
split_index = int(behaviors.height * 0.85)
behaviors_train = behaviors.slice(0, split_index)
behaviors_validation = behaviors.slice(split_index)
print(f"Size of train set: {behaviors_train.height}, from {behaviors_train['timestamp'].min()} to {behaviors_train['timestamp'].max()}")
print(f"Size of validation set: {behaviors_validation.height}, from {behaviors_validation['timestamp'].min()} to {behaviors_validation['timestamp'].max()}")

In [ ]:
behaviors_train.write_parquet("train/behaviors_train.parquet")
behaviors_validation.write_parquet("validation/behaviors_validation.parquet")

In [ ]:
news = pl.read_csv(
    "train/news.tsv", 
    separator="\t",
    quote_char=None,
    has_header=False,
    new_columns=["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
)
print(news.height)
news.head()

In [ ]:
def get_split_news_ids(df_behaviors: pl.DataFrame) -> set:
    # 1. Extract history IDs
    history_ids = (
        df_behaviors
        .select(
            pl.col("history")
            .drop_nulls()
            .str.split(" ")
            .explode(empty_as_null=True)
        )
        .filter(pl.col("history") != "")
        .rename({"history": "news_id"})
    )

    # 2. Extract impression IDs
    impression_ids = (
        df_behaviors
        .select(
            pl.col("impressions")
            .drop_nulls()
            .str.split(" ")
            .explode(empty_as_null=True)
        )
        .filter(pl.col("impressions") != "")
        .select(pl.col("impressions").str.extract(r"^([^-]+)", 1).alias("news_id"))
    )

    # Return as a set to avoid the is_in() ambiguity warning
    return set(
        pl.concat([history_ids, impression_ids])
        .unique()["news_id"]
        .to_list()
    )

# Extract unique ID sets
train_news_ids = get_split_news_ids(behaviors_train)
validation_news_ids = get_split_news_ids(behaviors_validation)

# Filter news dataframes without warnings
news_train = news.filter(pl.col("news_id").is_in(train_news_ids))
news_validation = news.filter(pl.col("news_id").is_in(validation_news_ids))

print(news_train.height)
print(news_validation.height)

In [ ]:
news_train.write_parquet("train/news_train.parquet")
news_validation.write_parquet("validation/news_validation.parquet")

### Read MIND Validation Dataset, Use as Testing

In [ ]:
behaviors_test = pl.read_csv(
    "test/behaviors.tsv", 
    separator="\t", 
    has_header=False, 
    new_columns=["session_id", "user_id", "timestamp", "history", "impressions"]
)
print(behaviors_test.height)
behaviors_test.head()

In [ ]:
behaviors_test = behaviors_test.with_columns(
    pl.col("timestamp").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p")
)
behaviors_test.head()

In [ ]:
behaviors_test.write_parquet("test/behaviors_test.parquet")

In [ ]:
news_test = pl.read_csv(
    "test/news.tsv", 
    separator="\t",
    quote_char=None,
    has_header=False,
    new_columns=["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
)
print(news_test.height)
news_test.head()

In [ ]:
news_test.write_parquet("test/news_test.parquet")